### Load

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import pandas as pd
from sqlalchemy import text
from data.common.db import make_engine

engine = make_engine()
docs = pd.read_sql(text("SELECT * FROM nlp.document"), engine)
kb = pd.read_sql(text("SELECT * FROM nlp.kb_entity"), engine)
print(len(docs), docs["article_title"].nunique())
docs.groupby("season").size()

3030 238


season
2015/16    291
2016/17    244
2017/18    258
2018/19    231
2019/20    116
2020/21    250
2021/22    257
2022/23    378
2023/24    319
2024/25    337
2025/26    251
2026/27     98
dtype: int64

### Length and sections

In [3]:
print(docs["n_chars"].describe().round(0))
docs["section"].value_counts().head(15)


count    3030.0
mean      589.0
std       426.0
min       200.0
25%       305.0
50%       462.0
75%       709.0
max      4048.0
Name: n_chars, dtype: float64


section
Introduction                 414
December                     149
EFL Cup                      136
April                        125
January                      124
September                    123
October                      121
August                       119
Pre-season                   119
FA Cup                       114
May                          113
November                     110
Pre-season and friendlies    107
February                     100
March                         98
Name: count, dtype: int64

### Read examples

In [4]:
for t in docs.sample(3, random_state=1)["text"]:
    print(t, "\n")
    

During the 2022–23 season, Manchester United's youth setup added several young players to its academy. The club brought in 14-year-old Louie Bradbury from Shrewsbury Town's youth setup and 13-year-old Zac Watson from Doncaster Rovers' academy, with both youngsters joining the club as part of its youth development system. 

On 21 March 2017, it was announced that Tottenham Hotspur would take part in the 2017 International Champions Cup playing against Paris Saint-Germain, Roma and Manchester City in the United States. On 4 July 2017, it was announced that Tottenham would welcome Juventus to play a friendly match in Wembley Stadium. Tottenham Hotspur played a closed door match in Hotspur Way against Leyton Orient on 12 July 2017. 

Chelsea returned to winning ways against Crystal Palace in a 2–0 victory at Wembley in the FA Cup semi-final to book a spot in the final against Liverpool. They returned to Premier League action with a 4–2 defeat to Arsenal at home on 20 April, with several de

### How often do known names appear?

In [5]:
sample = docs.sample(500, random_state=1)
full_names = set(kb.loc[(kb.entity_type == "PLAYER") & (kb.alias_kind == "full"), "alias"])
team_names = set(kb.loc[kb.entity_type == "TEAM", "alias"])
has_player = sample["text"].apply(lambda t: any(n in t for n in full_names))
has_team = sample["text"].apply(lambda t: any(n in t for n in team_names))
print(round(has_player.mean(), 2), round(has_team.mean(), 2))

0.59 0.94


###  Shared surnames

In [6]:
surnames = kb[kb.alias_kind == "surname"]
owners = surnames.groupby(["season", "alias"])["entity_id"].nunique()
shared = owners[owners > 1]
print(len(shared), "surnames shared by 2+ players in the same season")
shared.sort_values(ascending=False).head(10)

254 surnames shared by 2+ players in the same season


season   alias  
2018/19  Silva      4
2020/21  Mendy      3
2023/24  Silva      3
2016/17  Jones      3
2018/19  Ward       3
         Sánchez    3
2016/17  Davies     3
2021/22  Silva      3
2025/26  Gomes      3
2015/16  Wilson     3
Name: entity_id, dtype: int64